# Week 1, Lab 3 — Tools from scratch

No SDK. You describe tools in the prompt, parse the model's intent, run Python, send the result back.

This is what OpenAI Agents SDK, CrewAI, and LangChain are doing under the hood.


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [3]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


## 1. Setup


In [4]:
WEEK = 'Week 1'
LAB = 'Lab 3 — tools from scratch'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 1 / Lab 3 — tools from scratch
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [5]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn pydantic
else:
    %pip install -q ollama pydantic


## 2. Tools are just functions + a JSON schema


In [6]:
print("calculator(2+2) =", calculator("2+2"))
print("lookup_fact(mcp) =", lookup_fact("mcp"))
print("today_date() =", today_date())
print("schemas:", [t["name"] for t in TOOL_SCHEMAS])


calculator(2+2) = 4
lookup_fact(mcp) = The Model Context Protocol standardizes how agents connect to external tools and data sources.
today_date() = 2026-09-12
schemas: ['calculator', 'lookup_fact', 'today_date']


## 3. Teach the model the tool format


In [7]:
TOOLS_PREAMBLE = """You are a tool-using assistant.
If you need a tool, reply with ONLY JSON:
{"name": "<tool>", "arguments": {<args>}}
If you can answer without a tool, reply with normal text.

Tools:
- calculator(expression: str) — arithmetic only
- lookup_fact(topic: str) — local facts about agentic AI
- today_date() — today's date
"""

def run_one_step(user_text: str) -> str:
    reply = local_chat(
        [
            {"role": "system", "content": TOOLS_PREAMBLE},
            {"role": "user", "content": user_text},
        ],
        max_new_tokens=120,
        temperature=0.1,
    )
    print("MODEL:", reply)
    call = parse_tool_call(reply)
    if not call:
        return reply
    fn = {"calculator": calculator, "lookup_fact": lookup_fact, "today_date": today_date}.get(call["name"])
    if fn is None:
        return f"Unknown tool: {call['name']}"
    result = fn(**call["arguments"]) if call["arguments"] else fn()
    print("TOOL", call["name"], "->", result)
    final = local_chat(
        [
            {"role": "system", "content": "Answer the user using the tool result. Be brief."},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": reply},
            {"role": "user", "content": f"TOOL RESULT: {result}"},
        ],
        max_new_tokens=120,
        temperature=0.2,
    )
    return final

print("\n=== math ===")
print("FINAL:", run_one_step("What is 45 * 12 + 30?"))
print("\n=== fact ===")
print("FINAL:", run_one_step("What is MCP in one sentence?"))



=== math ===
Loading Hugging Face model Qwen/Qwen2.5-1.5B-Instruct on GPU ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new

MODEL: 780
FINAL: 780

=== fact ===
MODEL: MCP stands for Multi-Criteria Decision Analysis.
FINAL: MCP stands for Multi-Criteria Decision Analysis.


## 4. Exercise

1. Add a `reverse_text(text)` tool and ask the model to reverse a name.
2. Intentionally omit the JSON instruction and see how often tool-calling breaks.
3. Log `(thought?, tool, args, result)` as a list — that log is the ancestor of tracing in later SDKs.

**Next:** `lab4_simple_agent_loop.ipynb` — repeat Reason → Act → Observe until done.
